# Multi-Standard Overview
Load and compare OMOP CDM, FHIR R4, HL7 v2.5.1, and OpenEHR standard definitions.

In [1]:
import sys
sys.path.insert(0, "../../packages/sdk/src")

from portiere.models.target_model import get_target_model
from portiere.standards import list_standards, YAMLTargetModel

## Available Standards
List all registered standard definitions.

In [2]:
for name in list_standards():
    model = get_target_model(name)
    schema = model.get_schema()
    total_fields = sum(len(fields) for fields in schema.values())
    print(f"  {model.name} (v{model.version}) — {len(schema)} entities, {total_fields} fields")

  fhir_r4 (vR4) — 8 entities, 65 fields
  hl7v2_2.5.1 (v2.5.1) — 8 entities, 66 fields
  omop_cdm_v5.4 (vv5.4) — 8 entities, 69 fields


  openehr_1.0.4 (v1.0.4) — 7 entities, 47 fields


## Comparing Schemas
Compare entities and field counts across standards.

In [3]:
standards = ["omop_cdm_v5.4", "fhir_r4", "hl7v2_2.5.1", "openehr_1.0.4"]

for name in standards:
    model = get_target_model(name)
    schema = model.get_schema()
    print(f"\n{'='*50}")
    print(f"{model.name} ({model.standard_type})")
    print(f"{'='*50}")
    for entity, fields in schema.items():
        print(f"  {entity}: {len(fields)} fields")


omop_cdm_v5.4 (relational)
  person: 9 fields
  location: 9 fields
  visit_occurrence: 9 fields
  condition_occurrence: 6 fields
  drug_exposure: 11 fields
  measurement: 11 fields
  procedure_occurrence: 6 fields
  observation: 8 fields

fhir_r4 (resource)
  Patient: 10 fields
  Encounter: 8 fields
  Condition: 8 fields
  Observation: 10 fields
  MedicationRequest: 8 fields
  Procedure: 6 fields
  AllergyIntolerance: 7 fields
  DiagnosticReport: 8 fields

hl7v2_2.5.1 (segment)
  PID: 15 fields
  PV1: 10 fields
  DG1: 6 fields
  OBX: 9 fields
  OBR: 7 fields
  RXE: 10 fields
  AL1: 5 fields
  NK1: 4 fields



openehr_1.0.4 (archetype)
  demographics.person: 7 fields
  composition.encounter: 6 fields
  evaluation.problem_diagnosis: 8 fields
  instruction.medication_order: 8 fields
  observation.laboratory_test_result: 9 fields
  observation.blood_pressure: 4 fields
  action.procedure: 5 fields


## Target Descriptions for AI Mapping
Each standard includes embedding-friendly descriptions for semantic schema mapping.

In [4]:
for name in ["omop_cdm_v5.4", "fhir_r4"]:
    model = get_target_model(name)
    descs = model.get_target_descriptions()
    print(f"\n{model.name} — {len(descs)} descriptions (showing first 5):")
    for key, desc in list(descs.items())[:5]:
        print(f"  {key}: {desc}")


omop_cdm_v5.4 — 67 descriptions (showing first 5):
  person.person_id: person table: unique patient identifier number
  person.person_source_value: person table: original patient ID from source system
  person.gender_concept_id: person table: patient gender or biological sex
  person.year_of_birth: person table: patient birth year
  person.month_of_birth: person table: patient birth month

fhir_r4 — 65 descriptions (showing first 5):
  Patient.id: Patient: unique patient identifier
  Patient.identifier: Patient: business identifier for the patient (MRN, SSN, etc.)
  Patient.active: Patient: whether patient record is in active use (true/false)
  Patient.name: Patient: patient's full legal name
  Patient.telecom: Patient: contact information (phone, email)


## Source Patterns
Column-name patterns for fast schema matching.

In [5]:
model = YAMLTargetModel.from_name("omop_cdm_v5.4")
patterns = model.get_source_patterns()
print(f"OMOP CDM: {len(patterns)} source patterns (showing first 10):")
for pattern, (entity, field) in list(patterns.items())[:10]:
    print(f"  '{pattern}' → {entity}.{field}")

OMOP CDM: 77 source patterns (showing first 10):
  'patient_id' → person.person_id
  'subject_id' → person.person_id
  'person_id' → person.person_id
  'gender' → person.gender_concept_id
  'sex' → person.gender_concept_id
  'birth_date' → person.birth_datetime
  'date_of_birth' → person.birth_datetime
  'dob' → person.birth_datetime
  'race' → person.race_concept_id
  'ethnicity' → person.ethnicity_concept_id


## DDL Generation
Generate schema definitions in the native format for each standard.

In [6]:
for name in standards:
    model = get_target_model(name)
    ddl = model.generate_ddl()
    lines = ddl.split('\n')
    preview = '\n'.join(lines[:15])
    print(f"\n--- {model.name} DDL (first 15 lines) ---")
    print(preview)
    if len(lines) > 15:
        print(f"  ... ({len(lines) - 15} more lines)")


--- omop_cdm_v5.4 DDL (first 15 lines) ---
CREATE TABLE person (
    person_id INTEGER PRIMARY KEY,
    gender_concept_id INTEGER NOT NULL,
    year_of_birth INTEGER NOT NULL,
    month_of_birth INTEGER,
    day_of_birth INTEGER,
    birth_datetime TIMESTAMP,
    race_concept_id INTEGER,
    ethnicity_concept_id INTEGER,
    person_source_value VARCHAR(50)
);

CREATE TABLE location (
    location_id INTEGER PRIMARY KEY,
    address_1 VARCHAR(50),
  ... (83 more lines)



--- fhir_r4 DDL (first 15 lines) ---
{
  "Patient": {
    "resourceType": "Patient",
    "description": "Demographics and administrative information about a patient",
    "fields": {
      "id": {
        "type": "string",
        "required": true,
        "description": "Unique patient identifier"
      },
      "identifier": {
        "type": "Identifier[]",
        "required": false,
        "description": "Business identifier for the patient (MRN, SSN, etc.)"
      },
  ... (360 more lines)



--- hl7v2_2.5.1 DDL (first 15 lines) ---
# hl7v2_2.5.1 Segment Definitions


## PID — Patient Identification — demographics and identifiers
   PID-3 patient_id: Patient identifier list (MRN, SSN, etc.)
   PID-5 patient_name: Patient name (family, given, middle)
   PID-7 date_of_birth: Date/time of birth
   PID-8 administrative_sex: Administrative sex (F, M, O, U, A, N)
  PID-10 race: Race
  PID-11 patient_address: Patient address
  PID-13 phone_home: Phone number — home
  PID-14 phone_business: Phone number — business
  PID-15 primary_language: Primary language
  PID-16 marital_status: Marital status
  PID-19 ssn: Social security number
  ... (69 more lines)



--- openehr_1.0.4 DDL (first 15 lines) ---
# openehr_1.0.4 Archetype Definitions


## demographics.person
   Patient demographics (openEHR-DEMOGRAPHIC-PERSON)
                                                          /uid → uid: Unique person identifier
             /identities[at0000]/details[at0001]/items[at0002] → name: Person name
                                /details[at0001]/items[at0009] → date_of_birth: Date of birth
                                /details[at0001]/items[at0017] → sex: Administrative sex/gender
                                   /contacts[at0004]/addresses → address: Patient address
                     /contacts[at0004]/addresses/items[at0012] → telecom: Telephone or electronic communication
                                /details[at0001]/items[at0020] → ethnicity: Ethnicity

## composition.encounter
   Clinical encounter composition (openEHR-EHR-COMPOSITION.encounter)
  ... (55 more lines)


## Selecting Target Standard in Config

Configure which target standard to use in the pipeline. The `task` parameter declares
the project's purpose: `"standardize"` (default) for raw data → target standard, or
`"cross_map"` for transforming between standards.

In [7]:
from portiere.config import PortiereConfig

# Standardize projects — raw data → target standard (default task)
config_omop = PortiereConfig(target_model="omop_cdm_v5.4")
print(f"OMOP standardize: target={config_omop.target_model}")

config_fhir = PortiereConfig(target_model="fhir_r4")
print(f"FHIR standardize: target={config_fhir.target_model}")

config_hl7 = PortiereConfig(target_model="hl7v2_2.5.1")
print(f"HL7 v2 standardize: target={config_hl7.target_model}")

config_oehr = PortiereConfig(target_model="openehr_1.0.4")
print(f"OpenEHR standardize: target={config_oehr.target_model}")

# Cross-map project — transform between standards
# (task and source_standard are set via portiere.init(), not PortiereConfig)
print("\nCross-map projects use portiere.init(task='cross_map', source_standard=..., target_model=...)")

OMOP standardize: target=omop_cdm_v5.4
FHIR standardize: target=fhir_r4
HL7 v2 standardize: target=hl7v2_2.5.1
OpenEHR standardize: target=openehr_1.0.4

Cross-map projects use portiere.init(task='cross_map', source_standard=..., target_model=...)


## Custom Standard Definition
You can create custom standard definitions as YAML files.

```yaml
name: "my_custom_standard_v1"
version: "v1.0"
standard_type: "relational"
organization: "My Organization"

entities:
  patient:
    fields:
      patient_id:
        type: integer
        required: true
        description: "Unique patient identifier"
      name:
        type: string
        description: "Patient full name"
```

Load it with:
```python
model = get_target_model("custom:/path/to/my_custom_standard_v1.yaml")
```

---

## Running Pipelines with Different Target Standards and Engines

The real power of multi-standard support is running the **same pipeline** against different target standards. Below, we map the same source data to OMOP, FHIR, and OpenEHR using different engines.

### Same Source → OMOP CDM (Polars Engine)

Map hospital data to OMOP CDM using the default Polars engine. The `task="standardize"` parameter
(default) declares this as a raw-data-to-standard pipeline.

In [8]:
import portiere
from portiere.engines import PolarsEngine
from portiere.config import PortiereConfig, KnowledgeLayerConfig

# --- OMOP CDM pipeline with Polars ---
config_omop = PortiereConfig(
    target_model="omop_cdm_v5.4",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s"),
)

project_omop = portiere.init(
    name="Hospital → OMOP (Polars)",
    task="standardize",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config_omop,
)

# Run pipeline
source = project_omop.add_source("example_assets/14_multi_standard_overview/patients.csv", name="patients")
schema_map = project_omop.map_schema(source)
concept_map = project_omop.map_concepts(source=source)
etl_result = project_omop.run_etl(
    source,
    output_dir="./output_omop",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"OMOP CDM output (Polars): {len(etl_result.tables)} tables")
print(f"Engine: {project_omop.engine.engine_name}")
print(f"Task: standardize | Target: {project_omop.target_model}")

2026-04-16 00:30:18 [info     ] PolarsEngine initialized      


2026-04-16 00:30:18 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:30:18 [info     ] project.source_added           project='Hospital → OMOP (Polars)' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:30:21 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


...

2026-04-16 00:30:23 [info     ] Processing table               columns=6 table=person


2026-04-16 00:30:23 [info     ] Processing table               columns=4 table=location


2026-04-16 00:30:23 [info     ] ETL complete                   duration=0.004112 success=True


2026-04-16 00:30:23 [info     ] project.etl_complete           output_dir=./output_omop project='Hospital → OMOP (Polars)'


OMOP CDM output (Polars): 2 tables
Engine: polars
Task: standardize | Target: omop_cdm_v5.4


### Same Source → FHIR R4 (Spark Engine)

Map the same hospital data to FHIR R4 using SparkEngine for distributed processing.

In [9]:
try:
    from pyspark.sql import SparkSession
    from portiere.engines import SparkEngine

    spark = SparkSession.builder \
        .appName("portiere-fhir") \
        .master("local[*]") \
        .getOrCreate()

    project_fhir = portiere.init(
        name="Hospital → FHIR (Spark)",
        engine=SparkEngine(spark),
        target_model="fhir_r4",
    )
    print(f"FHIR Spark project: {project_fhir.name}")
    SPARK_AVAILABLE = True
except ImportError:
    SPARK_AVAILABLE = False
    spark = None
    project_fhir = None
    print("PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark")


PySpark not installed — Spark cells will be skipped. Install with: pip install pyspark


### Same Source → OpenEHR (Pandas Engine)

Map the same data to OpenEHR using PandasEngine for quick prototyping.

In [10]:
from portiere.engines import PandasEngine

# --- OpenEHR pipeline with Pandas ---
config_oehr = PortiereConfig(
    target_model="openehr_1.0.4",
    knowledge_layer=KnowledgeLayerConfig(backend="bm25s"),
)

project_oehr = portiere.init(
    name="Hospital → OpenEHR (Pandas)",
    task="standardize",
    engine=PandasEngine(),
    target_model="openehr_1.0.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config_oehr,
)

# Run pipeline
source = project_oehr.add_source("example_assets/14_multi_standard_overview/patients.csv", name="patients")
schema_map = project_oehr.map_schema(source)
concept_map = project_oehr.map_concepts(source=source)
etl_result = project_oehr.run_etl(
    source,
    output_dir="./output_openehr",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"OpenEHR output (Pandas): {len(etl_result.tables)} tables")
print(f"Engine: {project_oehr.engine.engine_name}")
print(f"Task: standardize | Target: {project_oehr.target_model}")

2026-04-16 00:30:23 [info     ] PandasEngine initialized      


2026-04-16 00:30:23 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:30:23 [info     ] project.source_added           project='Hospital → OpenEHR (Pandas)' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:30:24 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


...

2026-04-16 00:30:24 [info     ] Reading source                 format=csv path=example_assets/14_multi_standard_overview/patients.csv


2026-04-16 00:30:24 [info     ] Processing table               columns=6 table=demographics.person


2026-04-16 00:30:24 [info     ] ETL complete                   duration=0.003156 success=True


2026-04-16 00:30:24 [info     ] project.etl_complete           output_dir=./output_openehr project='Hospital → OpenEHR (Pandas)'


OpenEHR output (Pandas): 1 tables
Engine: pandas
Task: standardize | Target: openehr_1.0.4


### Cross-Mapping Between Pipeline Outputs

Once you have data in one standard, cross-map to others without re-running the full pipeline.
You can use standalone `CrossStandardMapper` directly, or initialize a `task="cross_map"` project
for tracked cross-mapping. Below uses the standalone mapper for quick conversion.

In [11]:
import pathlib
output_file = pathlib.Path("output_omop/person.csv")
if output_file.exists():
    import polars as pl
    df = pl.read_csv(output_file)
    print(df.head())
else:
    print(f"Output file not found (ETL must run first): {output_file}")


shape: (5, 6)
┌───────────┬─────────────────┬────────────────┬─────────────────┬────────────────┬────────────────┐
│ person_id ┆ person_source_v ┆ birth_datetime ┆ gender_concept_ ┆ race_concept_i ┆ ethnicity_conc │
│ ---       ┆ alue            ┆ ---            ┆ id              ┆ d              ┆ ept_id         │
│ str       ┆ ---             ┆ str            ┆ ---             ┆ ---            ┆ ---            │
│           ┆ str             ┆                ┆ str             ┆ str            ┆ str            │
╞═══════════╪═════════════════╪════════════════╪═════════════════╪════════════════╪════════════════╡
│ P001      ┆ James           ┆ 1958-03-14     ┆ M               ┆ White          ┆ Not Hispanic   │
│           ┆                 ┆                ┆                 ┆                ┆ or Latino      │
│ P002      ┆ Maria           ┆ 1972-08-22     ┆ F               ┆ White          ┆ Hispanic or    │
│           ┆                 ┆                ┆                 ┆           

### Engine Selection Guide

| Engine | Best For | Cross-Map Support | Pipeline Output |
|--------|----------|-------------------|-----------------|
| **PolarsEngine** | Local dev, up to ~10 GB | Polars DF in → Polars DF out | CSV, Parquet, JSON |
| **SparkEngine** | Large-scale (GB–TB), Databricks/EMR | Spark DF in → Spark DF out | Parquet, CSV, Delta |
| **PandasEngine** | Prototyping, small datasets | Pandas DF in → Pandas DF out | CSV, Parquet |

The engine only affects how data is read, transformed, and written during ETL. Cross-mapping preserves the DataFrame type of your engine pipeline.

```python
# Pattern: ETL with engine → cross-map stays in same DF type
project = portiere.init(engine=PolarsEngine(), target_model="omop_cdm_v5.4", ...)
etl_result = project.run_etl(source, output_dir="./omop_out")

# Cross-map OMOP output to FHIR — stays as Polars
import polars as pl
omop_df = pl.read_csv("./omop_out/person.csv")     # Polars read
fhir_df = project.cross_map("omop_cdm_v5.4", "fhir_r4", "person", omop_df)  # Polars out
fhir_df.write_parquet("./fhir_out/Patient.parquet")  # Polars write
```